In [3]:
import os
os.makedirs("src/DataScientist", exist_ok=True)
os.makedirs("src/Director", exist_ok=True)
os.makedirs("src/EndUser", exist_ok=True)

from data_transformation import (
    categorical_cols,
    create_train_test_split,
    load_adult_data,
    numeric_cols,
    split_X_y,
)

from model_factory import fit_and_score, model_dt, model_lr

In [ ]:
#Load data
data = load_adult_data(data_path='../data/adult.data', nrows = None )
X, y = split_X_y(data)
X_train, X_test, y_train, y_test = create_train_test_split(X, y, test_size=0.2, random_state=42)

In [57]:
X_sex_train, X_sex_test = X_train.drop(columns=['sex']), X_test.drop(columns=['sex'])
male_indexes = X_train[X_train["sex"] == " Male"].index.tolist()
female_indexes = X_train[X_train["sex"] == " Female"].index.tolist()

X_male, y_male = X_sex_train.loc[male_indexes], y_train.loc[male_indexes]
X_female, y_female = X_sex_train.loc[female_indexes], y_train.loc[female_indexes]

X_race = X.copy()
X_race = X_race.drop(columns=['race']);
X_race_train, X_race_test, y_race_train, y_race_test = create_train_test_split(X_race, y, test_size=0.2, random_state=42)


X_age = X.copy()
X_age = X_age.drop(columns=['age']);
X_age_train, X_age_test, y_age_train, y_age_test = create_train_test_split(X_age, y, test_size=0.2, random_state=42)


Gender analyisis

In [39]:
categorical_cols_sex = categorical_cols[0:6] + [categorical_cols[7]]

In [ ]:
dt_model = model_dt(categorical_cols_sex, numeric_cols)
lr_model = model_lr(categorical_cols_sex, numeric_cols)

dt_model.fit(X_sex_train, y_train);
lr_model.fit(X_sex_train, y_train);

In [ ]:
predictions = dt_model.predict(X_female)

print(sum(predictions == " <=50K"))
print(sum(predictions != " =50K"))

predictions = dt_model.predict(X_male)

print(sum(predictions == " <=50K"))
print(sum(predictions != " =50K"))

8151
8613
13885
17435


Finding the CAV using a logistic regression hyperplane

In [86]:
import numpy as np
import pandas as pd

# Combine male + female samples
X_concept = pd.concat([X_male, X_female], axis=0)

# Create concept labels
y_concept = np.concatenate([
    np.ones(len(X_male)),      # male = 1
    np.zeros(len(X_female))    # female = 0
])

cav_model = model_lr(categorical_cols_sex, numeric_cols)
cav_model.fit(X_concept, y_concept)

cav = cav_model[1].coef_[0]
v = cav / np.linalg.norm(cav)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,capital-gain,capital-loss,hours-pr-week,native-country
15738,32,Private,37210,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,0,0,45,United-States
9505,40,Local-gov,24763,Some-college,10,Divorced,Transport-moving,Unmarried,White,6849,0,40,United-States
26417,24,Private,113936,Bachelors,13,Never-married,Prof-specialty,Own-child,White,0,0,40,United-States
14701,51,Private,237630,HS-grad,9,Married-civ-spouse,Tech-support,Husband,White,7298,0,50,United-States
18530,44,Private,310255,Some-college,10,Married-civ-spouse,Craft-repair,Husband,White,0,0,60,United-States
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12792,57,Private,231232,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,0,0,40,United-States
13832,53,Private,149217,HS-grad,9,Married-civ-spouse,Craft-repair,Husband,White,0,0,40,Puerto-Rico
15369,40,Self-emp-inc,57233,HS-grad,9,Married-civ-spouse,Exec-managerial,Husband,White,0,0,60,United-States
23650,55,Private,82098,HS-grad,9,Married-civ-spouse,Exec-managerial,Husband,Asian-Pac-Islander,0,0,55,United-States


In [80]:
len(X_male.iloc[0])

13

In [78]:
len(v)

32

In [ ]:
X_test